In [1]:
!pip install -U transformers==4.45.2 sentencepiece

In [2]:
# Install compatible versions
!pip install -q transformers==4.45.2 sentencepiece

# Import libraries
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
import torch

# Check if GPU is available
device = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Using device: {device}')

if device == 'cpu':
    print('WARNING: No GPU detected.')
    print('Go to Runtime > Change runtime type > GPU')

# Load pretrained TrOCR model
processor = TrOCRProcessor.from_pretrained(
    'microsoft/trocr-base-printed'
)

model = VisionEncoderDecoderModel.from_pretrained(
    'microsoft/trocr-base-printed'
)

# Move model to GPU
model = model.to(device)

# Configure model for generation
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size

print('Model loaded successfully!')
print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

VisionEncoderDecoderModel has generative capabilities, as `prepare_inputs_for_generation` is explicitly overwritten. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.
Some weights of VisionEncoderDecoderModel were not initialized from the model checkpoint at microsoft/trocr-base-printed and are newly initialized: ['enco

generation_config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded successfully!
Model parameters: 333,921,792


In [4]:
!unzip -q data.zip

In [17]:
!unzip data.zip

Archive:  data.zip
replace data/labels.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
  inflating: data/labels.csv         
 extracting: data/raw/other/35.png   
 extracting: data/raw/other/48.png   
 extracting: data/raw/other/16.png   
 extracting: data/raw/other/23.png   
 extracting: data/raw/other/44.png   
 extracting: data/raw/other/37.png   
 extracting: data/raw/other/29.png   
 extracting: data/raw/other/13.png   
 extracting: data/raw/other/24.png   
 extracting: data/raw/other/38.png   
 extracting: data/raw/other/11.png   
 extracting: data/raw/other/12.png   
 extracting: data/raw/other/8.png    
 extracting: data/raw/other/14.png   
 extracting: data/raw/other/42.png   
 extracting: data/raw/other/36.png   
 extracting: data/raw/other/9.png    
 extracting: data/raw/other/25.png   
 extracting: data/raw/other/6.png    
 extracting: data/raw/other/40.png   
 extracting: data/raw/other/22.png   
 extracting: data/raw/other/30.png   
 extracting: data/raw/other/49.png   
 ext

In [18]:
import os

print(os.path.exists("data/raw/other/0.png"))

True


In [26]:
# Train/Test Split

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = torch.utils.data.random_split(
    dataset,
    [train_size, test_size]
)

print(f"Training Samples: {len(train_dataset)}")
print(f"Testing Samples: {len(test_dataset)}")

Training Samples: 144
Testing Samples: 36


In [27]:
from torch.utils.data import DataLoader
from transformers import AdamW

# Create data loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=4
)

# Optimizer
optimizer = AdamW(
    model.parameters(),
    lr=5e-5
)

print(f"Training batches per epoch: {len(train_loader)}")
print("Ready to train!")

Training batches per epoch: 36
Ready to train!


In [25]:
# Fix signboard folder name
dataset.data["image"] = dataset.data["image"].str.replace(
    "data/raw/signboard/",
    "data/raw/signboards/",
    regex=False
)

# Check all image files again
import os

missing_files = [
    path for path in dataset.data["image"]
    if not os.path.exists(path)
]

print("Missing files:", len(missing_files))

if len(missing_files) == 0:
    print("All image files found successfully!")
else:
    print(missing_files[:20])

Missing files: 0
All image files found successfully!


In [28]:
num_epochs = 3

for epoch in range(num_epochs):

    model.train()
    total_loss = 0

    print(f"\nEpoch {epoch + 1}/{num_epochs}")
    print("-" * 30)

    for batch_idx, batch in enumerate(train_loader):

        pixel_values = batch["pixel_values"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            pixel_values=pixel_values,
            labels=labels
        )

        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        if batch_idx % 10 == 0:
            print(
                f"Batch {batch_idx}/{len(train_loader)} "
                f"| Loss: {loss.item():.4f}"
            )

    avg_loss = total_loss / len(train_loader)

    print(
        f"Epoch {epoch + 1} complete "
        f"| Average Loss: {avg_loss:.4f}"
    )

print("\nTraining complete!")


Epoch 1/3
------------------------------
Batch 0/36 | Loss: 17.3515
Batch 10/36 | Loss: 5.9144
Batch 20/36 | Loss: 3.9175
Batch 30/36 | Loss: 3.8286
Epoch 1 complete | Average Loss: 5.3414

Epoch 2/3
------------------------------
Batch 0/36 | Loss: 3.6216
Batch 10/36 | Loss: 3.6291
Batch 20/36 | Loss: 3.6333
Batch 30/36 | Loss: 3.6140
Epoch 2 complete | Average Loss: 3.6576

Epoch 3/3
------------------------------
Batch 0/36 | Loss: 3.5668
Batch 10/36 | Loss: 3.4517
Batch 20/36 | Loss: 3.6054
Batch 30/36 | Loss: 3.5444
Epoch 3 complete | Average Loss: 3.6019

Training complete!


In [29]:
# Evaluate the model on test data

model.eval()

total_test_loss = 0

with torch.no_grad():

    for batch in test_loader:

        pixel_values = batch["pixel_values"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            pixel_values=pixel_values,
            labels=labels
        )

        total_test_loss += outputs.loss.item()

average_test_loss = total_test_loss / len(test_loader)

print(f"Test Loss: {average_test_loss:.4f}")

Test Loss: 3.5598


In [31]:
model.eval()

sample = test_dataset[0]

pixel_values = sample["pixel_values"].unsqueeze(0).to(device)

with torch.no_grad():
    generated_ids = model.generate(
        pixel_values,
        max_new_tokens=128
    )

predicted_text = processor.batch_decode(
    generated_ids,
    skip_special_tokens=True
)[0]

print("Predicted Text:")
print(predicted_text)

Predicted Text:
��������������������������������������������������������������������������������������������������������������������������������


In [32]:
model.eval()

sample = test_dataset[0]

# Actual text
actual_text = test_dataset.dataset.data.iloc[
    test_dataset.indices[0]
]["text"]

# Image
pixel_values = sample["pixel_values"].unsqueeze(0).to(device)

# Prediction
with torch.no_grad():
    generated_ids = model.generate(
        pixel_values,
        max_new_tokens=128
    )

predicted_text = processor.batch_decode(
    generated_ids,
    skip_special_tokens=True
)[0]

print("ACTUAL TEXT:")
print(actual_text)

print("\nPREDICTED TEXT:")
print(predicted_text)

ACTUAL TEXT:
میں سے ایک ٹرک اور گراؤنڈ زیرو کا 20 ٹن

PREDICTED TEXT:
��������������������������������������������������������������������������������������������������������������������������������


In [34]:
model.eval()

print("=== Model Evaluation on Test Images ===")
print()

correct = 0
total = 0

with torch.no_grad():

    for batch in test_loader:

        pixel_values = batch["pixel_values"].to(device)
        labels = batch["labels"]

        # Replace -100 with pad token before decoding
        labels_for_decode = labels.clone()
        labels_for_decode[
            labels_for_decode == -100
        ] = processor.tokenizer.pad_token_id

        # Generate prediction
        generated_ids = model.generate(
            pixel_values,
            max_new_tokens=128
        )

        generated_text = processor.batch_decode(
            generated_ids,
            skip_special_tokens=True
        )

        actual_text = processor.batch_decode(
            labels_for_decode,
            skip_special_tokens=True
        )

        for pred, actual in zip(generated_text, actual_text):

            total += 1

            if pred.strip() == actual.strip():
                correct += 1

            print(f"Predicted: {pred}")
            print(f"Actual: {actual}")
            print()

accuracy = (correct / total) * 100 if total > 0 else 0

print(f"Accuracy: {accuracy:.1f}% ({correct}/{total} correct)")

=== Model Evaluation on Test Images ===

Predicted: ��������������������������������������������������������������������������������������������������������������������������������
Actual: میں سے ایک ٹرک اور گراؤنڈ زیرو کا 20 ٹن

Predicted: ��������������������������������������������������������������������������������������������������������������������������������
Actual: پر یقین رکھنے والی جماعت ہے، اس کی صفوں میں مذہبی

Predicted: ��������������������������������������������������������������������������������������������������������������������������������
Actual: نبی صلی اللہ علیہ وآلہ وسلم کے امیج کو دنیا میں خراب کر رہے ہیں۔ الطاف

Predicted: ��������������������������������������������������������������������������������������������������������������������������������
Actual: علی اور نبی آخرالزماں حضرت محمدمصطفے ﷺ سے جاملتاہے اور 

Predicted: ��������������������������������������������������������������������������������������������������������������������������������
Actual

In [36]:
save_path = '/content/drive/MyDrive/SI26-urdu-ocr-model'

model.save_pretrained(save_path)
processor.save_pretrained(save_path)

print(f"Model saved to Google Drive: {save_path}")
print("You can load this model again next week without retraining.")

Model saved to Google Drive: /content/drive/MyDrive/SI26-urdu-ocr-model
You can load this model again next week without retraining.
